# 07 Kerala climate: long-term rain and model projections

Question (T142 only): how has Kerala rainfall varied since 1981, and what do the climate models
project for 2021-2050 against 1981-2010?

Observed: S08 IMD 0.25 degree daily rain, annual totals of the Kerala cell box mean (8.25-12.75 N,
74.75-77.5 E; it also holds Tamil Nadu and Karnataka border cells, so it is a box, not Kerala's
boundary), 1981-2024. S08 is a gauge-based gridded reconstruction and is local only (IMD forbids
reproduction): only derived totals are exported.

Projected: S06 Open-Meteo climate API, seven CMIP6 HighResMIP models, daily, bias-corrected to
ERA5-Land (the default), at two Kerala points. The API names no scenario ("as close to RCP8.5 as
possible", D107), so none is claimed. Each model gives its own 2021-2050 vs 1981-2010 change in
mean annual rain, over complete years only (EC_Earth3P_HR has no 2050). **The spread across
models is model (and scenario) spread, not forecast confidence**: no probability or CI is
attached, and no model year is a forecast of any real year. Request budget: two 30-year requests
for two points (about 2,200 weighted calls each by the documented weighting, under the 10,000 a
day limit), fetched once; re-runs read the stored responses.

Run from the repository root in a fresh Python 3.12 process. Outputs go to
`reports/eda/07_climate/` (or `ATLAS_EDA_OUT`).

In [ ]:
import csv
import json
import os
import statistics
import textwrap
from collections import defaultdict
from datetime import UTC, date, datetime
from pathlib import Path
from urllib.parse import urlencode

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from atlas.analysis.quality import caption
from atlas.analysis.replay import daily_box_mean
from atlas.config import load_state
from atlas.contracts import Budget, FetchContext, RawArtifact
from atlas.features.m05 import year_artifacts
from atlas.ingest import s06
from atlas.ingest.http import Fetcher
from atlas.ingest.raw_store import RawStore
from atlas.ingest.s08 import BOXES, S08

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/07_climate"))
OUT.mkdir(parents=True, exist_ok=True)
RAW = Path("data/raw")
YEARS = range(1981, 2025)
WINDOWS = {"1981-2010": (1981, 2010), "2021-2050": (2021, 2050)}
POINTS = {"Thiruvananthapuram": (8.5, 76.95), "Kozhikode": (11.25, 75.78)}
kl = load_state(Path("configs/states/kl.yaml"))


def write_csv(name, rows):
    with open(OUT / name, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)


def s06_window(first, last):
    adapter = s06.S06("kl", list(POINTS.values()), date(first, 1, 1), date(last, 12, 31))
    url = f"{s06.API}?{urlencode(sorted(adapter.params().items()))}"
    entry = RawStore(RAW).latest_ok(url)
    if entry is None:  # one-off fetch; every later run reads the stored response
        adapter.fetcher = Fetcher(RawStore(RAW), s06.HOSTS)
        ctx = FetchContext(
            source_id="S06",
            state="kl",
            run_id=f"t142-s06-{first}",
            requested_at=datetime.now(UTC),
            raw_root=RAW,
            budget=Budget(max_requests=1, max_bytes=64 * 1024**2),
        )
        raw = adapter.fetch(ctx)[0]
    else:
        a = entry["artifact"]
        raw = RawArtifact.model_validate(a | {"path": Path(a["path"])})
    return raw, adapter.parse(raw, kl.source("S06"))

In [ ]:
raws = year_artifacts(RAW, YEARS)
annual = []
for raw in raws:
    year = int(raw.source_url.rsplit("=", 1)[1])
    means = daily_box_mean(S08("kl", [year]).parse(raw, kl.source("S08")))
    days = 366 if year % 4 == 0 else 365
    assert len(means) == days, f"{year}: {len(means)} days with values"
    annual.append({"year": year, "rain_mm": round(sum(m for m, _ in means.values()), 1)})
write_csv("s08_kerala_annual.csv", annual)
base_obs = statistics.fmean(r["rain_mm"] for r in annual if r["year"] <= 2010)
print(len(annual), "years; 1981-2010 mean", round(base_obs), "mm")

In [ ]:
fetched, totals = [], defaultdict(list)  # (point, model, window) -> annual totals
for window, (first, last) in WINDOWS.items():
    raw, table = s06_window(first, last)
    fetched.append(raw.fetched_at)
    cells = list(dict.fromkeys(table.column("cell_id").to_pylist()))
    names = dict(zip(cells, POINTS, strict=True))  # responses keep the request order
    year_rain = defaultdict(list)
    for c, d, m, r in zip(
        *(table.column(k).to_pylist() for k in ("cell_id", "date_local", "model", "rain_mm")),
        strict=True,
    ):
        year_rain[names[c], m, d.year].append(r)
    for (p, m, _y), values in sorted(year_rain.items()):
        if None not in values:  # complete years only (EC_Earth3P_HR has no 2050)
            totals[p, m, window].append(sum(values))
rows = []
for p in POINTS:
    for m in s06.MODELS:
        b, f = totals[p, m, "1981-2010"], totals[p, m, "2021-2050"]
        rows.append(
            {
                "point": p,
                "model": m,
                "base_years": len(b),
                "future_years": len(f),
                "base_mm": round(statistics.fmean(b), 1),
                "future_mm": round(statistics.fmean(f), 1),
                "change_pct": round((statistics.fmean(f) / statistics.fmean(b) - 1) * 100, 1),
            }
        )
write_csv("s06_model_change.csv", rows)
for r in rows:
    print(r)

In [ ]:
s08_vintage = f"IMD year files fetched {min(r.fetched_at for r in raws):%Y-%m-%d}"
s06_vintage = f"Open-Meteo climate API, fetched {min(fetched):%Y-%m-%d}"
cap1 = caption(
    f"S08 IMD 0.25 deg grid (reconstruction; {s08_vintage}), Kerala cell box {BOXES['kl']}",
    "annual box-mean rain",
    "mm",
    f"{YEARS[0]}-01-01",
    f"{YEARS[-1]}-12-31",
    f"{len(annual)} complete years",
)
fig, ax = plt.subplots(figsize=(7, 4))
fig.subplots_adjust(bottom=0.2)
ax.plot([r["year"] for r in annual], [r["rain_mm"] for r in annual], marker="o", ms=3)
ax.axhline(base_obs, color="k", lw=0.8, label=f"1981-2010 mean {base_obs:.0f} mm")
ax.set_ylabel("Annual rain (mm)")
ax.set_title("Kerala box annual rain, observed (gridded reconstruction)", fontsize=10)
ax.legend(fontsize=7)
fig.text(0.01, 0.01, textwrap.fill(cap1, 110), fontsize=5)
fig.savefig(OUT / "s08_annual_rain.png", dpi=80, metadata={"Software": None})
plt.close(fig)

cap2 = caption(
    f"S06 CMIP6 HighResMIP via {s06_vintage}; bias-corrected to ERA5-Land; no named scenario",
    "change in mean annual rain per model, 2021-2050 vs 1981-2010 (projection, not a forecast; "
    "spread across models is not forecast confidence)",
    "%",
    "1981-01-01",
    "2050-12-31",
    f"{len(s06.MODELS)} models x {len(POINTS)} points",
)
fig, ax = plt.subplots(figsize=(7, 4))
fig.subplots_adjust(bottom=0.22, right=0.72)
for i, m in enumerate(s06.MODELS):
    ys = [r["change_pct"] for r in rows if r["model"] == m]
    ax.scatter([j + (i - 3) * 0.04 for j in range(len(POINTS))], ys, s=18, label=m)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(range(len(POINTS)), list(POINTS))
ax.set_ylabel("Change in mean annual rain (%)")
ax.set_title("Model projections, one dot per model (spread is not confidence)", fontsize=10)
ax.legend(fontsize=6, loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.text(0.01, 0.01, textwrap.fill(cap2, 110), fontsize=5)
fig.savefig(OUT / "s06_model_change.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
changes = [r["change_pct"] for r in rows]
summary = {
    "s08": {
        "years": [YEARS[0], YEARS[-1]],
        "mean_1981_2010_mm": round(base_obs, 1),
        "min_mm": min(r["rain_mm"] for r in annual),
        "max_mm": max(r["rain_mm"] for r in annual),
        "first_fetch": min(r.fetched_at for r in raws).isoformat(),
    },
    "s06": {
        "points": POINTS,
        "models": list(s06.MODELS),
        "scenario": None,
        "bias_corrected": True,
        "change_pct_min": min(changes),
        "change_pct_max": max(changes),
        "change_pct_median": statistics.median(changes),
        "fetched": [t.isoformat() for t in fetched],
    },
    "charts": [
        {
            "file": "s08_annual_rain.png",
            "kind": "descriptive",
            "ci": None,
            "source": "S08 IMD 0.25 deg gridded rainfall",
            "vintage": s08_vintage,
        },
        {
            "file": "s06_model_change.png",
            "kind": "projection",
            "ci": None,  # model spread is shown as dots, never as a confidence interval
            "source": "S06 Open-Meteo CMIP6 HighResMIP",
            "vintage": s06_vintage,
        },
    ],
    "limits": [
        "Projections, not forecasts: no model year is a forecast of a real year.",
        "Spread across models is model and scenario spread, not forecast confidence.",
        "The API names no scenario; none is claimed (D107).",
        "Two points only (request budget); they are not area means.",
        "S08 is a gridded reconstruction; the cell box is not Kerala's boundary.",
        "EC_Earth3P_HR has no 2050: its future mean uses 2021-2049.",
    ],
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", "utf-8", newline="\n")
print(json.dumps(summary, indent=2))